# Bonus Notebook — Build On Your Own Agent

This notebook is **optional** and **self-paced**. If you finish notebooks 0–4 early, work through
whichever exercises below interest you, in any order. Each one is independent — pick and choose.

Every exercise reuses tools and patterns you've already built today: `tavily_search`, `think_tool`,
`AgentState`, `Command`, and the compiled `full_agent` graph from notebook 4. Nothing new to install,
no new concepts to learn — just new ways to combine what you already have.

> 💡 Each exercise has a 💡 solution you can expand if you get stuck. Try first, peek second.


---
## ⚙️ Setup

Run this first — it loads everything you built in notebooks 0–4.


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from typing_extensions import Literal

from deep_research_from_scratch.utils import tavily_search, think_tool, get_today_str
from deep_research_from_scratch.state_scope import AgentState, AgentInputState
from deep_research_from_scratch.research_agent_scope import clarify_with_user, write_research_brief
from deep_research_from_scratch.research_agent import researcher_agent
from deep_research_from_scratch.research_agent_full import deep_researcher_builder, run_research, final_report_generation

checkpointer = InMemorySaver()
full_agent = deep_researcher_builder.compile(checkpointer=checkpointer)

print("✅ Everything loaded — full_agent is ready to use.")


---
## 🏋️ Exercise 1 — Write your own tool (easy)

> **Goal:** Practice the `@tool` pattern from notebook 3 by writing a brand new tool from scratch.

So far the research agent only has two tools: `tavily_search` and `think_tool`. Let's give it a third.

**Task:** Write a tool called `word_count` that takes a piece of text and returns how many words it contains.
Follow the exact same pattern as `think_tool` in `utils.py` — a docstring with an `Args:` section,
decorated with `@tool(parse_docstring=True)`.


In [ ]:
# TODO: write the word_count tool
@tool(parse_docstring=True)
def word_count(text: str) -> str:
    """???

    Args:
        text: ???

    Returns:
        ???
    """
    # YOUR CODE HERE
    pass

# Test it
print(word_count.invoke({"text": "AI tools for rehabilitation exercise tracking"}))


<details>
<summary>💡 Solution (click to expand)</summary>

```python
@tool(parse_docstring=True)
def word_count(text: str) -> str:
    """Count the number of words in a piece of text.

    Args:
        text: The text to count words in

    Returns:
        A string reporting the word count
    """
    count = len(text.split())
    return f"Word count: {count}"

print(word_count.invoke({"text": "AI tools for rehabilitation exercise tracking"}))
# Word count: 6
```

**Why this matters:** every tool you'll ever give an agent follows this exact shape — a typed function,
a docstring LangChain reads to build the schema, and a return value the LLM can read back. You just
proved you can extend the agent's abilities without touching any of the graph logic.

</details>


---
## 🏋️ Exercise 2 — Give the research agent your new tool (medium)

> **Goal:** Wire a custom tool into the actual research loop and watch the LLM decide whether to use it.

Having a tool isn't enough — the agent needs to know it exists. In notebook 3, tools are bound to the
model with `model.bind_tools(tools)`. Let's rebuild a tiny version of that loop with `word_count` added.

**Task:** Build a one-shot agent call (not a full graph — just a single `invoke`) that gives the model
access to `tavily_search`, `think_tool`, **and** your new `word_count` tool. Ask it something where
`word_count` might plausibly get used, e.g. "Summarize the history of LangGraph in exactly one paragraph,
then tell me how many words your summary has."


In [ ]:
# TODO: bind all three tools to a model and invoke it
tools = [tavily_search, think_tool, word_count]
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.0)
model_with_tools = model.bind_tools(tools)

response = ...  # YOUR CODE HERE: invoke model_with_tools with a HumanMessage

print(f"Tool calls: {[tc['name'] for tc in response.tool_calls]}")


<details>
<summary>💡 Solution (click to expand)</summary>

```python
tools = [tavily_search, think_tool, word_count]
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.0)
model_with_tools = model.bind_tools(tools)

response = model_with_tools.invoke([
    HumanMessage(content="Summarize the history of LangGraph in exactly one paragraph, "
                          "then tell me how many words your summary has.")
])

print(f"Tool calls: {[tc['name'] for tc in response.tool_calls]}")
```

**Reflection:** did the model actually call `word_count`, or did it just count the words itself in its
head? LLMs are often capable of doing the math without the tool — this is a good moment to notice that
giving a model a tool doesn't guarantee it'll use it. The prompt matters as much as the tool itself.

</details>


---
## 🏋️ Exercise 3 — Add a new field to `AgentState` (medium-hard)

> **Goal:** Extend the pipeline's shared state with your own field, the same way `research_brief`,
> `notes`, and `final_report` were added in notebook 2.

Let's say you want the pipeline to also track **how many sources were used** in the final report.

**Task:**
1. Define a new `TypedDict` called `ExtendedAgentState` that extends `AgentState` with one new field:
   `source_count: int`.
2. Write a tiny standalone node, `count_sources`, that reads `state["notes"]`, counts how many `"---
   SOURCE"` markers appear across all notes combined (that's how `tavily_search`'s output is formatted —
   check `utils.py`'s `format_search_output` if you want a reminder), and returns `{"source_count": ...}`.
3. You do **not** need to wire this into the real graph — just prove the node works by calling it
   directly on a fake state dict.


In [ ]:
# TODO: define ExtendedAgentState and the count_sources node
class ExtendedAgentState(AgentState):
    source_count: int

def count_sources(state) -> dict:
    # YOUR CODE HERE
    pass

# Test with fake state
fake_state = {
    "notes": [
        "--- SOURCE 1: App X ---\ncontent...\n--- SOURCE 2: App Y ---\ncontent...",
        "--- SOURCE 1: App Z ---\ncontent..."
    ]
}
print(count_sources(fake_state))


<details>
<summary>💡 Solution (click to expand)</summary>

```python
class ExtendedAgentState(AgentState):
    source_count: int

def count_sources(state) -> dict:
    notes = state.get("notes", [])
    combined = "\n".join(notes)
    count = combined.count("--- SOURCE")
    return {"source_count": count}

fake_state = {
    "notes": [
        "--- SOURCE 1: App X ---\ncontent...\n--- SOURCE 2: App Y ---\ncontent...",
        "--- SOURCE 1: App Z ---\ncontent..."
    ]
}
print(count_sources(fake_state))
# {'source_count': 3}
```

**Why this matters:** this is exactly the pattern behind every field you've seen in `AgentState` all
day — pick a name, pick a type, write a node that reads existing state and writes the new field. State
classes aren't fixed once defined; you extend them the same way every time.

</details>


---
## 🏋️ Exercise 4 — Add a routing branch with `Command` (hard)

> **Goal:** Practice the `Command(goto=..., update=...)` pattern from notebook 2's `clarify_with_user`,
> but with a routing decision you design yourself.

In notebook 4, `run_research` always goes straight to `final_report_generation` — there's no decision
to make. Let's add one.

**Task:** Write a node called `check_report_length` that runs **after** `final_report_generation`. It
should:
- Read `state["final_report"]`
- If the report is shorter than 200 characters, route back to `final_report_generation` to try again
  (update nothing — just retry)
- Otherwise, route to `END`

You don't need to rebuild the whole graph — just write the function and reason through what it would
return for a short fake report vs. a long one.


In [ ]:
# TODO: write check_report_length
def check_report_length(state: AgentState) -> Command[Literal["final_report_generation", "__end__"]]:
    # YOUR CODE HERE
    pass

# Test with two fake states
short_report_state = {"final_report": "Too short."}
long_report_state   = {"final_report": "A" * 500}

print(check_report_length(short_report_state))
print(check_report_length(long_report_state))


<details>
<summary>💡 Solution (click to expand)</summary>

```python
def check_report_length(state: AgentState) -> Command[Literal["final_report_generation", "__end__"]]:
    report = state.get("final_report", "")
    if len(report) < 200:
        return Command(goto="final_report_generation", update={})
    return Command(goto=END, update={})

short_report_state = {"final_report": "Too short."}
long_report_state   = {"final_report": "A" * 500}

print(check_report_length(short_report_state))  # routes back to retry
print(check_report_length(long_report_state))   # routes to END
```

**Why this matters:** this is the same shape as `clarify_with_user` — a node that doesn't just update
state, it decides where the graph goes next. The only difference is the condition being checked. If you
wanted to wire this into the real graph, you'd add it as a node and point `final_report_generation`'s
edge at it instead of straight to `END` — try that for extra credit if you have time left.

</details>


---
## 🏋️ Exercise 5 — Design your own mini research agent (open-ended)

> **Goal:** No scaffolding this time — design something small end-to-end using only what you've learned.

Pick **one** of these (or invent your own, equally-sized idea):

- A 3-node graph that takes a topic, searches for it once with `tavily_search`, and writes a 2-sentence
  summary — no clarification step, no loop, just START → search → summarize → END.
- A tool-equipped single-node agent (like Exercise 2) that has a system prompt instructing it to always
  call `think_tool` before answering, and test whether it actually follows that instruction.
- A new `AgentState` field + node pair (like Exercise 3) for something *you* think would be useful to
  track in a research pipeline — word count of the brief, number of clarification rounds, anything.

There's no solution provided for this one — bring your idea to one of the instructors if you want a
second pair of eyes on it.


In [ ]:
# Your design here — start from a blank cell, reuse any imports from the Setup cell above.

